# Test MLP Medium Ensemble

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import torch
import torch.nn as nn

TEST_PATH=Path("data/test_features-1.csv")
PREPROCESSOR_PATH=Path("models/preprocessor_v2.joblib")
CONFIG_PATH=Path("models/medium_ensemble/config.joblib")
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
print("Device:",DEVICE)

Device: cpu


In [4]:
test_df=pd.read_csv(TEST_PATH)
assert "Id" in test_df.columns, "El test debe contener la columna Id."
ids=test_df["Id"].copy()
X_raw=test_df.drop(columns=[c for c in ["Id","SalePrice"] if c in test_df.columns])
preprocessor=joblib.load(PREPROCESSOR_PATH)
X_test=preprocessor.transform(X_raw)
if hasattr(X_test,"toarray"): X_test=X_test.toarray()
X_test=np.asarray(X_test,dtype=np.float32)
print("Test:",test_df.shape,"Procesado:",X_test.shape)

Test: (292, 80) Procesado: (292, 278)


In [5]:
class MLP(nn.Module):
    def __init__(self,input_dim,hidden,dropout=0.0):
        super().__init__(); layers=[]; prev=input_dim
        for h in hidden:
            layers += [nn.Linear(prev,h),nn.ReLU()]
            if dropout>0: layers.append(nn.Dropout(dropout))
            prev=h
        layers.append(nn.Linear(prev,1)); self.net=nn.Sequential(*layers)
    def forward(self,x): return self.net(x).squeeze(1)

config=joblib.load(CONFIG_PATH)
seeds=config["seeds"]; hidden=config["hidden_layers"]; hp=config["hyperparams"]
all_preds=[]
Xt=torch.tensor(X_test,dtype=torch.float32).to(DEVICE)
for seed in seeds:
    ckpt=torch.load(f"models/medium_ensemble/medium_seed_{seed}.pt",map_location=DEVICE,weights_only=False)
    model=MLP(X_test.shape[1],hidden,hp["dropout"]).to(DEVICE)
    model.load_state_dict(ckpt["model_state_dict"]); model.eval()
    with torch.no_grad(): scaled=model(Xt).cpu().numpy()
    pred=scaled*float(ckpt["target_std"])+float(ckpt["target_mean"])
    all_preds.append(pred)
    print(f"Cargado seed {seed}")
pred_final=np.mean(np.stack(all_preds,axis=0),axis=0)
print("Predicciones listas:",pred_final.shape)

Cargado seed 42
Cargado seed 123
Cargado seed 456
Cargado seed 789
Cargado seed 2026
Predicciones listas: (292,)


## Generar archivo de entrega

In [6]:
predictions=pd.DataFrame({"Id":ids.to_numpy(),"Prediction":pred_final})
assert predictions.columns.tolist()==["Id","Prediction"]
assert len(predictions)==len(test_df)
assert predictions["Prediction"].notna().all()
predictions.to_csv("predictions.csv",index=False)
print("Archivo listo: predictions.csv")
print("Columnas:",predictions.columns.tolist())
display(predictions.head())

Archivo listo: predictions.csv
Columnas: ['Id', 'Prediction']


,Id,Prediction
0,893,154943.890625
1,1106,341251.593750
2,414,99198.085938
3,523,172828.921875
4,1037,352010.312500
